# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [3]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [5]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_15548\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


In [6]:
y

date
1960-01-01   -0.8
1960-02-01   -1.1
1960-03-01   -0.2
1960-04-01    0.0
1960-05-01    0.0
             ... 
2025-04-01    0.3
2025-05-01    0.2
2025-06-01    0.0
2025-07-01    0.0
2025-08-01    0.1
Freq: MS, Name: UNRATE, Length: 788, dtype: float64

# Run_config

In [7]:

# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# ---------- Paramètres bagging ----------
use_bagging = True
B_boot = 30
L_block = 12
rng = np.random.default_rng(123)

# ---------- Paramètres conformal (comme code 1) ----------
use_conformal = True
step_size = 12
pi_windows = 3
alpha = 0.05  # 95% => q = quantile 1-alpha des erreurs

## Boostrap Utilities

In [8]:
# =========================
# cellule 2 : Bootstrap + fonctions Conformal
# =========================
def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à longueur n."""
    n = len(arr)
    if L <= 0 or L > n:
        raise ValueError("L_block invalide")
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def bagged_h_forecast_AR1(y_tr, h, trend, B, L, rng):
    """
    Prévision à horizon h par bagging (residual moving-block bootstrap) pour AR(1).
    Retourne (yhat_mean, yhat_dist, base_pred)
    """
    base_model = AutoReg(y_tr, lags=1, old_names=False, trend=trend).fit()
    base_fc = base_model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base_model.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné aux résidus

    boot_preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)
        y_b = fitted + res_b
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=1, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        boot_preds.append(float(fc_b.iloc[-1]))

    return float(np.mean(boot_preds)), np.array(boot_preds), base_pred

def fit_predict_ar_p(y_tr, h, trend="c", p=1):
    """Fit AR(p) sur y_tr, retourne la prévision au pas h (dernier point)."""
    m = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    fc = m.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    return float(fc.iloc[-1])

def conformal_q_from_past_windows_pos(y, i_end, *, h=12, step_size=12, pi_windows=3, trend="c", p=1, alpha=0.05):
    """
    Version robuste (sans test 'date in index'):
    - i_end = position de t_end dans y.index
    - fenêtres de calibration: i_cal = i_end - k*step_size
    - cible à comparer: i_cal + h
    - erreurs: |y[i_cal+h] - yhat( train jusqu'à i_cal )|
    """
    errs = []
    for k in range(1, pi_windows + 1):
        i_cal = i_end - k * step_size
        i_cal_fore = i_cal + h
        if i_cal < 0 or i_cal_fore >= len(y):
            continue

        y_tr_cal = y.iloc[: i_cal + 1]
        if len(y_tr_cal) < max(36, p + 2):  # cohérent avec min_train_n
            continue

        yhat_cal = fit_predict_ar_p(y_tr_cal, h=h, trend=trend, p=p)
        err = abs(float(y.iloc[i_cal_fore]) - yhat_cal)
        errs.append(err)

    if len(errs) == 0:
        return np.nan

    # intervalle symétrique: q = quantile(1-alpha) des erreurs absolues
    return float(np.quantile(errs, 1 - alpha))

In [7]:
# =========================
# cellule 3 : Sécurisation de la série y (index strictement début de mois)
# =========================
y = pd.Series(y.astype(float).values, index=pd.to_datetime(y.index))

# Aligne sur le 1er du mois (start of month) de façon robuste
y.index = y.index.to_period("M").to_timestamp(how="start")

# Fixe la fréquence MS (Month Start)
y = y.asfreq("MS").dropna()

print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS


In [8]:
# =========================
# cellule 4 : Boucle pseudo-OOS continue + prédiction + intervalles (conformal ou bootstrap)
# =========================
rows = []
last_model = None
last_fit_end = None

# on boucle uniquement sur des t_end qui ont un t_end+h existant
for i_end, t_end in enumerate(y.index[:-h]):
    y_tr = y.iloc[: i_end + 1]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # fit AR(p) base
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # ----- Prévision à h mois (bagging ou base) -----
    if use_bagging:
        yhat_h, yhat_dist, yhat_h_base = bagged_h_forecast_AR1(
            y_tr=y_tr, h=h, trend=trend, B=B_boot, L=L_block, rng=rng
        )
    else:
        fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_h_base = yhat_h
        yhat_dist = None

    # t_fore pris DIRECTEMENT depuis l'index (robuste)
    t_fore = y.index[i_end + h]
    y_true = float(y.iloc[i_end + h])

    # ----- Intervalles -----
    if use_conformal:
        q = conformal_q_from_past_windows_pos(
            y=y, i_end=i_end, h=h, step_size=step_size, pi_windows=pi_windows,
            trend=trend, p=p_fixed, alpha=alpha
        )
        if np.isfinite(q):
            yhat_p05 = yhat_h - q
            yhat_p95 = yhat_h + q
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan
    else:
        # fallback: quantiles bootstrap (si bagging), sinon NA
        if use_bagging and (yhat_dist is not None) and len(yhat_dist) > 0:
            yhat_p05 = float(np.percentile(yhat_dist, 5))
            yhat_p95 = float(np.percentile(yhat_dist, 95))
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan

    rows.append((t_fore, yhat_h, y_true, yhat_p05, yhat_p95, yhat_h_base))

In [9]:
# =========================
# cellule 5 : DataFrame OOS
# =========================
if rows:
    df_oos_ar1 = (
        pd.DataFrame(
            rows,
            columns=["date", "y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"]
        )
        .set_index("date")
        .sort_index()
    )
else:
    df_oos_ar1 = pd.DataFrame(columns=["y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"])
    df_oos_ar1.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_ar1)}")
print(df_oos_ar1.head(-3))


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  y_hat_p05  y_hat_p95  y_hat_base
date                                                          
1963-12-01  0.070473     0.0        NaN        NaN   -0.080890
1964-01-01  0.017682    -0.1        NaN        NaN    0.141077
1964-02-01  0.090722    -0.5        NaN        NaN    0.408114
1964-03-01  0.165681    -0.3        NaN        NaN    0.242637
1964-04-01  0.091963    -0.4        NaN        NaN    0.238955
...              ...     ...        ...        ...         ...
2025-01-01 -0.010382     0.3  -3.300127   3.279363    0.043861
2025-02-01 -0.006875     0.2  -3.275618   3.261868    0.072590
2025-03-01 -0.019816     0.3  -2.895811   2.856179    0.101443
2025-04-01  0.005002     0.3  -0.589461   0.599465    0.130432
2025-05-01 -0.003852     0.2  -0.617284   0.609580    0.102350

[738 rows x 5 columns]


In [10]:
# ---------- (facultatif) Scores par période ----------
if len(df_oos_ar1):
    df_val  = df_oos_ar1.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_ar1.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"]))
        r2   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")

    if len(df_test):
        mae  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"]))
        r2   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")


📊 Validation 83–89 — n=84 | MAE=0.817 | RMSE=1.234 | R²=-0.949
📊 Test 90–2025 — n=428 | MAE=0.867 | RMSE=1.600 | R²=-0.100


In [11]:
# ==========================================
# Sauvegardes — AR(1) bagging + conformal (h=12)
# ==========================================
import joblib
import pickle

AR1_LAST_PKL  = "AR1_last_trained_model.pkl"
AR1_LAST_META = "AR1_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_h12_oos_bundle.pkl"

In [12]:
if last_model is not None:
    try:
        joblib.dump(last_model, AR1_LAST_PKL)
        print(f"💾 Modèle AR(1) sauvegardé → {AR1_LAST_PKL}")
    except Exception:
        with open(AR1_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(1) sauvegardé (pickle) → {AR1_LAST_PKL}")

💾 Modèle AR(1) sauvegardé → AR1_last_trained_model.pkl


In [13]:
bundle = {
    # ---------- Prévisions OOS ----------
    "oos_predictions": (
        df_oos_ar1
        .reset_index()
        .rename(columns={
            "y_hat": "y_pred",
            "y_true": "y_obs"
        })
        .assign(
            date=lambda d: pd.to_datetime(d["date"])
                            .dt.to_period("M")
                            .dt.to_timestamp(how="start")
        )
    ),

    # ---------- Paramètres du modèle ----------
    "params": {
        "model": "AR(1)",
        "trend": trend,
        "horizon": h,
        "lags": p_fixed,
        "min_train_n": min_train_n,

        # ---- bagging ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block),

        # ---- conformal ----
        "use_conformal": bool(use_conformal),
        "pi_windows": int(pi_windows),
        "step_size": int(step_size),
        "alpha": float(alpha),
    },

    # ---------- Métadonnées ----------
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_ar1)),
        "n_intervals_available": int(df_oos_ar1["y_hat_p05"].notna().sum()),
        "first_interval_date": (
            str(df_oos_ar1["y_hat_p05"].first_valid_index().date())
            if df_oos_ar1["y_hat_p05"].notna().any()
            else None
        ),
    }
}

with open(AR1_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)

print(f"💾 Bundle AR(1) OOS sauvegardé → {AR1_BUNDLE}")


💾 Bundle AR(1) OOS sauvegardé → AR1_h12_oos_bundle.pkl


In [14]:
meta_row = {
    "model": "AR(1)",
    "trend": trend,
    "horizon": h,
    "lags": p_fixed,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_ar1)),
    "use_bagging": bool(use_bagging),
    "use_conformal": bool(use_conformal),
    "pi_windows": pi_windows if use_conformal else None,
    "step_size": step_size if use_conformal else None,
    "n_intervals_available": int(df_oos_ar1["y_hat_p05"].notna().sum()),
}

pd.DataFrame([meta_row]).to_csv(AR1_LAST_META, index=False)
print(f"💾 Méta AR(1) sauvegardée → {AR1_LAST_META}")

💾 Méta AR(1) sauvegardée → AR1_last_trained_model_meta.csv


# Graphique

In [15]:
# =========================
# cellule 7 : Préparer df_obs + df_fcst (format utilsforecast)
# =========================
from utilsforecast.plotting import plot_series

SERIES_ID = "UNRATE"  # adapte si tu veux (ex: "UNRATE_stationary")

# df_obs : observations
df_obs = (
    df_oos_ar1
    .reset_index()
    .rename(columns={"date": "ds", "y_true": "y"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "y"]]
)

# df_fcst : forecasts + intervalles
df_fcst = (
    df_oos_ar1
    .reset_index()
    .rename(columns={"date": "ds", "y_hat": "AR1", "y_hat_p05": "AR1-lo-95", "y_hat_p95": "AR1-hi-95"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "AR1", "AR1-lo-95", "AR1-hi-95"]]
)

print(df_obs.head(2))
print(df_fcst.head(2))

  unique_id         ds    y
0    UNRATE 1963-12-01  0.0
1    UNRATE 1964-01-01 -0.1
  unique_id         ds       AR1  AR1-lo-95  AR1-hi-95
0    UNRATE 1963-12-01  0.070473        NaN        NaN
1    UNRATE 1964-01-01  0.017682        NaN        NaN


In [27]:
import pandas as pd
import plotly.graph_objects as go
from utilsforecast.plotting import plot_series

# =========================
# cellule 9 : Zoom + cadrage de l’axe Y
# =========================
START_ZOOM = "1990-01-01"
END_ZOOM   = "2025-08-01"

segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2019-fin"),
]

# Sécurité datetime
df_obs["ds"] = pd.to_datetime(df_obs["ds"])
df_fcst["ds"] = pd.to_datetime(df_fcst["ds"])

start_zoom_dt = pd.to_datetime(START_ZOOM)
end_zoom_dt   = pd.to_datetime(END_ZOOM)

df_obs_z = df_obs.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")
df_fcst_z = df_fcst.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")

fig = plot_series(
    df=df_obs_z,
    forecasts_df=df_fcst_z,
    level=[95],
    engine="plotly",
).update_layout(
    height=450,
    yaxis=dict(range=[-10, 12])
)

# =========================
# Renommage de la légende
# =========================
for trace in fig.data:
    name = trace.name.lower()
    if trace.name == "y":
        trace.name = "Observed"
    elif trace.name == "AR1":
        trace.name = "AR(1) forecast"
    elif "95" in name or "lo" in name or "hi" in name:
        trace.name = "Conformal Prediction"

# =========================
# Un seul bouton "Segments" (sans 1990)
# =========================
ymin, ymax = -10, 12
SEG_GROUP = "SEGMENTS"

fig.update_layout(legend=dict(groupclick="togglegroup"))

first = True
for i, (start, _, _) in enumerate(segments):

    # 👇 on ignore le premier segment (1990)
    if i == 0:
        continue

    x = pd.to_datetime(start)

    if not (start_zoom_dt <= x <= end_zoom_dt):
        continue

    fig.add_trace(
        go.Scatter(
            x=[x, x],
            y=[ymin, ymax],
            mode="lines",
            legendgroup=SEG_GROUP,
            name="Segments" if first else None,
            showlegend=first,
            visible="legendonly",
            line=dict(color="gray", width=1, dash="dash"),
            hoverinfo="skip",
        )
    )
    first = False

fig.show()

Nous utilisons une méthode de Conformal Prediction séquentielle (Split CP) qui calibre l’incertitude à partir des erreurs passées observées dans des fenêtres temporelles comparables.
Les bandes sont volontairement discontinues car elles reflètent les moments où l’incertitude est statistiquement estimable.

## Est-ce que le problème vaut la peine d’être modélisé ?

### Phase 1 : 
Le taux de chômage américain suit un cycle économique d’environ 9,5 ans, correspondant au cycle long.
Une récession apparaît au début des années 1990, puis au début des années 2000 (éclatement de la bulle Internet). La crise des subprimes survient environ 8 ans plus tard, suivie de la crise du Covid-19 11 ans après. L'impact a été immense durant la crise Covid-19 mais il est vite résorbé. 

### Phase 2 : 
Le modèle **AR(1)** utilisé est volontairement très basique. Il suit essentiellement la dynamique passée du taux de chômage. Pourtant, ce **baseline simple** permet déjà de mettre en évidence des informations clés.

En période de stabilité, le modèle auto-régressif parvient globalement à **capter la direction de l’évolution du chômage**, bien qu'il reste loin des observations. Il saisit correctement la dynamique générale, même de manière approximative. Le sens de la croissance du chômage dans la crise aussi est très bien captée par ce baseline. Ces deux faits nous donnent un signal fort. La structure temporelle existe et peut être exploitée.

### Phase 3 : 
L'ajout du Conformal Prediction nous permet de nuancer ces résultats. 
Lors des **ruptures structurelles**, le comportement change nettement. Le modèle parvient à détecter les **pics de chômage**, mais la **qualité des prévisions se dégrade fortement**. Cette dégradation se reflète directement dans l’**élargissement des intervalles conformes**. Ils  traduisent une incertitude croissante du modèle.

Un point crucial apparaît alors : **la perte de la capacité d’anticipation**. Par exemple, les effets de la crise de 2008 deviennent visibles dès la seconde moitié de l’année, mais le modèle ne les capte qu’à partir d’**octobre 2009**. Le signal est détecté, mais trop tard pour une décision politique.

### En résumé : 
En résumé, le modèle capte correctement la **direction des variations du chômage**, mais sa **fiabilité diminue fortement en période de crise**. L’intérêt majeur est qu’il **ne masque pas cette incertitude**. Au contraire, il la rend visible. Cela confirme que le problème mérite d’être modélisé. 

### Direction de l'expérimentation
Si vous étiez dans le membre de l'analyste, que conseillerez-vous pour continuer la direction de l'analyse?

# Perspective 1 : Sur la justesse de l'intensité
Nous avons vu que le comportement n'est pas le même durant la stabilité et les crises. L'évaluation de la modélisation se base donc au moins sur 02 Ou 03 périodes : 
- 1990 à 2008 ; 
- Seconde moitié de 2008 à 2019 ; 
- 2019 à maintenant ; 

Pour un souci de bonne répartition du temps, nous avons donc séparer la première période en deux sous-périodes de 1990 à 2000 puis 2000 à 2008. 

# Perspective 2 : sur la capacité d'anticipation
Le modèle réagit avec retard car il n’utilise que le chômage du mois passé (Lag = 1) pour anticiper celui du mois présent. 

Faut-t-il donc augmenter le nombre de retards pris en compte par le modèle pour mieux prévoir? Ou bien certaines situations, comme les crises, relèvent-elles de chocs extérieurs, nécessitant l’intégration d’autres indicateurs que le seul historique du chômage pour être anticipées ?

Faisons alors d'une part une analyse de l'auto-corrélation du chômage pour identifier le nombre de retards optimal. 
D'autre part, analysons les crises des Etats-Unis d'amérique en profondeur. 